In [ ]:
# Mount Google Drive (if running on Colab)
# try:
#     from google.colab import drive
#     drive.mount('/content/drive')
#     print("Google Drive mounted successfully!")
# except ImportError:
#     print("Not running on Colab - using local paths")

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import math
import os
import random
from tqdm.auto import tqdm

# Reproducability Code
def seed_everything(seed=1234):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    torch.use_deterministic_algorithms(True)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

seed_everything(1234)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- CONFIGURATION ---
SEQ_LEN = 30
BATCH_SIZE = 64
EPOCHS = 50
LEARNING_RATE = 1e-3

# --- DATA LOADING & PREPROCESSING ---
def load_and_scale_data(train_path, test_path):
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
    # Features to scale (exclude id, cycle, and RUL)
    feature_cols = [c for c in train_df.columns if c not in ['id', 'cycle', 'RUL']]
    
    scaler = MinMaxScaler()
    train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
    test_df[feature_cols] = scaler.transform(test_df[feature_cols])
    
    return train_df, test_df, feature_cols

def create_sequences(df, feature_cols, seq_len):
    X, y = [], []
    for engine_id, group in df.groupby('id'):
        data = group[feature_cols].values
        labels = group['RUL'].values
        
        # Slide window
        for i in range(len(data) - seq_len + 1):
            X.append(data[i:i + seq_len])
            y.append(labels[i + seq_len - 1])
            
    return np.array(X), np.array(y)

def create_dataloader(X, y, shuffle=True):
    dataset = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32))
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, worker_init_fn=seed_worker)

# --- MODEL DEFINITION ---
class RULTransformer(nn.Module):
    def __init__(self, input_dim, d_model=64, nhead=4, num_layers=2, dropout=0.1):
        super(RULTransformer, self).__init__()
        self.input_linear = nn.Linear(input_dim, d_model)
        
        self.pos_encoder = nn.Parameter(torch.zeros(1, SEQ_LEN, d_model))
        
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.regressor = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )
        
    def forward(self, src):
        # src shape: [batch_size, seq_len, input_dim]
        src = self.input_linear(src) + self.pos_encoder
        output = self.transformer_encoder(src)
        # Take the output of the last time step for sequence classification/regression
        output = output[:, -1, :]
        return self.regressor(output).squeeze(1)

In [ ]:
def train_and_evaluate(train_path, test_path, model_save_path):
    train_full_df, test_df, features = load_and_scale_data(train_path, test_path)
    
    val_ids = train_full_df['id'].unique()[-20:]
    train_df = train_full_df[~train_full_df['id'].isin(val_ids)]
    val_df = train_full_df[train_full_df['id'].isin(val_ids)]
    
    X_train, y_train = create_sequences(train_df, features, SEQ_LEN)
    X_val, y_val = create_sequences(val_df, features, SEQ_LEN)
    X_test, y_test = create_sequences(test_df, features, SEQ_LEN)
    
    train_loader = create_dataloader(X_train, y_train, shuffle=True)
    val_loader = create_dataloader(X_val, y_val, shuffle=False)
    test_loader = create_dataloader(X_test, y_test, shuffle=False)
    
    model = RULTransformer(input_dim=len(features)).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    best_val_rmse = float('inf')
    final_train_rmse = 0
    
    for epoch in range(EPOCHS):
        model.train()
        train_losses = []
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)
        for batch_X, batch_y in pbar:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            predictions = model(batch_X)
            loss = criterion(predictions, batch_y)
            loss.backward()
            optimizer.step()
            
            train_losses.append(loss.item())
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})
            
        train_rmse = math.sqrt(np.mean(train_losses))
        
        model.eval()
        val_preds, val_targets = [], []
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X = batch_X.to(device)
                preds = model(batch_X)
                val_preds.extend(preds.cpu().numpy())
                val_targets.extend(batch_y.numpy())
                
        val_rmse = math.sqrt(mean_squared_error(val_targets, val_preds))
        
        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            torch.save(model.state_dict(), model_save_path)
            final_train_rmse = train_rmse
            
    model.load_state_dict(torch.load(model_save_path))
    model.eval()

    test_preds, test_targets = [], []
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X = batch_X.to(device)
            preds = model(batch_X)
            test_preds.extend(preds.cpu().numpy())
            test_targets.extend(batch_y.numpy())
            
    test_rmse = math.sqrt(mean_squared_error(test_targets, test_preds))
    
    return {'Train RMSE': final_train_rmse, 'Val RMSE': best_val_rmse, 'Test RMSE': test_rmse}

# Store results by seed and dataset
all_results = {}

In [ ]:
# 0. Train & Test on `linear_rul_no_norm_0` with multiple seeds
import os

# Detect if running on Colab and set base path accordingly
try:
    from google.colab import drive
    # If you mounted drive, use this path
    BASE_DATA_DIR = '/content/drive/MyDrive/sutd_50039_deep_learning/data/processed-nasa-data/data_cleaning_1'
except ImportError:
    # Local path (when running locally)
    BASE_DATA_DIR = '../data/processed-nasa-data/data_cleaning_1'

SEEDS = [42, 1234, 999]
dataset_name_0 = 'FD001_linear_rul_no_norm_0'
train_file_0 = os.path.join(BASE_DATA_DIR, 'linear_rul_no_norm_0/train_processed_rul_only_fd001.csv')
test_file_0 = os.path.join(BASE_DATA_DIR, 'linear_rul_no_norm_0/test_processed_rul_only_fd001.csv')

all_results[dataset_name_0] = {}

for seed in SEEDS:
    seed_everything(seed)
    print(f"\n{'='*60}")
    print(f"Training {dataset_name_0} with SEED={seed}")
    print(f"{'='*60}")
    
    model_save_path = f'best_transformer_no_norm_seed_{seed}.pth'
    metrics = train_and_evaluate(train_file_0, test_file_0, model_save_path)
    all_results[dataset_name_0][f'seed_{seed}'] = metrics
    
    print(f"Seed {seed} Results -> Train RMSE: {metrics['Train RMSE']:.2f} | Val RMSE: {metrics['Val RMSE']:.2f} | Test RMSE: {metrics['Test RMSE']:.2f}")

In [ ]:
# 1. Train & Test on `linear_rul_1` with multiple seeds
dataset_name_1 = 'linear_rul_1'
train_file_1 = os.path.join(BASE_DATA_DIR, 'linear_rul_1/train_processed_rul_only_fd001.csv')
test_file_1 = os.path.join(BASE_DATA_DIR, 'linear_rul_1/test_processed_rul_only_fd001.csv')

all_results[dataset_name_1] = {}

for seed in SEEDS:
    seed_everything(seed)
    print(f"\n{'='*60}")
    print(f"Training {dataset_name_1} with SEED={seed}")
    print(f"{'='*60}")
    
    model_save_path = f'best_transformer_norm_1_seed_{seed}.pth'
    metrics = train_and_evaluate(train_file_1, test_file_1, model_save_path)
    all_results[dataset_name_1][f'seed_{seed}'] = metrics
    
    print(f"Seed {seed} Results -> Train RMSE: {metrics['Train RMSE']:.2f} | Val RMSE: {metrics['Val RMSE']:.2f} | Test RMSE: {metrics['Test RMSE']:.2f}")

In [ ]:
# 2. Train & Test on `piecewise_rul_2` with multiple seeds
dataset_name_2 = 'FD001_piecewise_rul_2'
train_file_2 = os.path.join(BASE_DATA_DIR, 'piecewise_rul_2/train_processed_rul_piecewise_150_fd001.csv')
test_file_2 = os.path.join(BASE_DATA_DIR, 'piecewise_rul_2/test_processed_rul_piecewise_150_fd001.csv')

all_results[dataset_name_2] = {}


for seed in SEEDS:
    seed_everything(seed)
    print(f"\n{'='*60}")
    print(f"Training {dataset_name_2} with SEED={seed}")
    print(f"{'='*60}")
    
    model_save_path = f'best_transformer_norm_1_seed_{seed}.pth'
    metrics = train_and_evaluate(train_file_2, test_file_2, model_save_path)
    all_results[dataset_name_2][f'seed_{seed}'] = metrics
    
    print(f"Seed {seed} Results -> Train RMSE: {metrics['Train RMSE']:.2f} | Val RMSE: {metrics['Val RMSE']:.2f} | Test RMSE: {metrics['Test RMSE']:.2f}")


In [ ]:
# 3. Train & Test on 'low_variance_1' with multiple seeds  
dataset_name_3 = 'FD001_low_variance_1'
BASE_FE_DIR = '/content/drive/MyDrive/sutd_50039_deep_learning/data/processed-nasa-data/feature_engineering_2' if 'BASE_DATA_DIR' in dir() and '/content' in BASE_DATA_DIR else '../data/processed-nasa-data/feature_engineering_2'
train_file_3 = os.path.join(BASE_FE_DIR, 'low_variance_1/train_fd001_low_variance_1.csv')
test_file_3 = os.path.join(BASE_FE_DIR, 'low_variance_1/test_fd001_low_variance_1.csv')

all_results[dataset_name_3] = {}

for seed in SEEDS:
    seed_everything(seed)
    print(f"\n{'='*60}")
    print(f"Training {dataset_name_3} with SEED={seed}")
    print(f"{'='*60}")
    
    model_save_path = f'best_transformer_low_variance_seed_{seed}.pth'
    metrics = train_and_evaluate(train_file_3, test_file_3, model_save_path)
    all_results[dataset_name_3][f'seed_{seed}'] = metrics
    
    print(f"Seed {seed} Results -> Train RMSE: {metrics['Train RMSE']:.2f} | Val RMSE: {metrics['Val RMSE']:.2f} | Test RMSE: {metrics['Test RMSE']:.2f}")


In [ ]:
# 4. Train & Test on `linear_rul_no_norm_0` with multiple seeds
dataset_name_4 = 'FD002_linear_rul_no_norm_0'
train_file_4 = os.path.join(BASE_DATA_DIR, 'linear_rul_no_norm_0/train_processed_rul_only_fd002.csv')
test_file_4 = os.path.join(BASE_DATA_DIR, 'linear_rul_no_norm_0/test_processed_rul_only_fd002.csv')

all_results[dataset_name_4] = {}

for seed in SEEDS:
    seed_everything(seed)
    print(f"\n{'='*60}")
    print(f"Training {dataset_name_4} with SEED={seed}")
    print(f"{'='*60}")
    
    model_save_path = f'best_transformer_no_norm_seed_{seed}.pth'
    metrics = train_and_evaluate(train_file_4, test_file_4, model_save_path)
    all_results[dataset_name_4][f'seed_{seed}'] = metrics
    
    print(f"Seed {seed} Results -> Train RMSE: {metrics['Train RMSE']:.2f} | Val RMSE: {metrics['Val RMSE']:.2f} | Test RMSE: {metrics['Test RMSE']:.2f}")

In [ ]:
# 5. Train & Test on `linear_rul_1` with multiple seeds
dataset_name_5 = 'FD002_linear_rul_1'
train_file_5 = os.path.join(BASE_DATA_DIR, 'linear_rul_1/train_processed_rul_only_fd002.csv')
test_file_5 = os.path.join(BASE_DATA_DIR, 'linear_rul_1/test_processed_rul_only_fd002.csv')

all_results[dataset_name_5] = {}

for seed in SEEDS:
    seed_everything(seed)
    print(f"\n{'='*60}")
    print(f"Training {dataset_name_5} with SEED={seed}")
    print(f"{'='*60}")
    
    model_save_path = f'best_transformer_norm_1_seed_{seed}.pth'
    metrics = train_and_evaluate(train_file_5, test_file_5, model_save_path)
    all_results[dataset_name_5][f'seed_{seed}'] = metrics
    
    print(f"Seed {seed} Results -> Train RMSE: {metrics['Train RMSE']:.2f} | Val RMSE: {metrics['Val RMSE']:.2f} | Test RMSE: {metrics['Test RMSE']:.2f}")

In [ ]:
# 6. Train & Test on `piecewise_rul_2` with multiple seeds
dataset_name_6 = 'FD002_piecewise_rul_2'
train_file_6 = os.path.join(BASE_DATA_DIR, 'piecewise_rul_2/train_processed_rul_piecewise_150_fd002.csv')
test_file_6 = os.path.join(BASE_DATA_DIR, 'piecewise_rul_2/test_processed_rul_piecewise_150_fd002.csv')

all_results[dataset_name_6] = {}


for seed in SEEDS:
    seed_everything(seed)
    print(f"\n{'='*60}")
    print(f"Training {dataset_name_6} with SEED={seed}")
    print(f"{'='*60}")
    
    model_save_path = f'best_transformer_norm_1_seed_{seed}.pth'
    metrics = train_and_evaluate(train_file_6, test_file_6, model_save_path)
    all_results[dataset_name_6][f'seed_{seed}'] = metrics
    
    print(f"Seed {seed} Results -> Train RMSE: {metrics['Train RMSE']:.2f} | Val RMSE: {metrics['Val RMSE']:.2f} | Test RMSE: {metrics['Test RMSE']:.2f}")


In [ ]:
# Display detailed results and compute averages
print(f"\n{'='*80}")
print("DETAILED RESULTS BY SEED")
print(f"{'='*80}\n")

for dataset_name, seed_results in all_results.items():
    print(f"\n### {dataset_name.upper()} ###")
    df_results = pd.DataFrame(seed_results).T
    print(df_results.to_string())
    
    print(f"\n--- MEAN METRICS ---")
    mean_metrics = df_results.mean()
    for metric_name, value in mean_metrics.items():
        print(f"{metric_name}: {value:.2f}")

print(f"\n{'='*80}")
print("COMPARISON TABLE - ALL DATASETS & SEEDS")
print(f"{'='*80}\n")

# Create a comprehensive comparison table
comparison_data = []
for dataset_name, seed_results in all_results.items():
    for seed_label, metrics in seed_results.items():
        seed_num = seed_label.split('_')[1]
        comparison_data.append({
            'Dataset': dataset_name,
            'Seed': seed_num,
            'Train RMSE': metrics['Train RMSE'],
            'Val RMSE': metrics['Val RMSE'],
            'Test RMSE': metrics['Test RMSE']
        })

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

print(f"\n{'='*80}")
print("AVERAGES ACROSS ALL SEEDS")
print(f"{'='*80}\n")

# Create summary table with averages
summary_data = []
for dataset_name in all_results.keys():
    seed_dfs = [pd.DataFrame([metrics]).T for metrics in all_results[dataset_name].values()]
    avg_df = pd.concat(seed_dfs, axis=1).mean(axis=1)
    summary_data.append({
        'Dataset': dataset_name,
        'Mean Train RMSE': avg_df['Train RMSE'],
        'Mean Val RMSE': avg_df['Val RMSE'],
        'Mean Test RMSE': avg_df['Test RMSE']
    })

summary_df = pd.DataFrame(summary_data)
print("\n=== SUMMARY TABLE ===")
print(summary_df.to_string(index=False))

# Save comparison results to CSV
print(f"\n{'='*80}")
print("SAVING RESULTS TO CSV")
print(f"{'='*80}\n")

# Save detailed results
comparison_df.to_csv('transformer_results_detailed.csv', index=False)
print("✓ Saved: transformer_results_detailed.csv")

# Save summary results
summary_df.to_csv('transformer_results_summary.csv', index=False)
print("✓ Saved: transformer_results_summary.csv")

print("\nFiles saved in the current working directory.")